In [ ]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.clear_highlights()
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', which='upper', style='red')
pp.add_highlight(which='lower gap', style='bold blue')

pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v_', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

In [ ]:
import statetracker as st
# Initialize manager
with st.Manager() as mgr:
    
    # Define leaf counters
    mut_state = st.State(5, name='mut_state')
    del_state = st.State(5, name='del_state')
    ins_site_state = st.State(2, name='ins_site_state')
    ins_position_state = st.State(5, name='ins_position_state')
    v_state = st.State(2).named('v_state')
    
    # Build composite counters
    ins_state = st.product([ins_position_state, ins_site_state],
        name='ins_state')
    cre_state = st.stack([mut_state, del_state, ins_state],
        name='cre_state')
    seq_state = st.product([cre_state, v_state], name='seq_state')
    
    bc_state = st.synced_to(seq_state).named('bc_state')

    # I want this 
    rows_state = st.State(6, name='rows_state')
    cols_state = st.State(8, name='cols_state')
    plate_state = st.product([rows_state, cols_state]).named('plate_state')
    
    st.sync(plate_state, seq_state)
    
# Print DAG
plate_state.print_dag('minimal')

# Print sequences
plate_state.get_iteration_df().reset_index()
